## 1. Imports and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("unified_features_v2.csv")
print(f"Total rows loaded: {len(df)}")
print(df["traffic_source"].value_counts())
df.head()

## 2. Separate SDCC (Display-Only)

SDCC uses the SCOOT sensor system; DCC/DLR use SCATS. Mean traffic volume: SDCC=142 vs DCC=701 (~5x difference). Root cause unclear (genuine low traffic or incompatible systems). Decision: exclude SDCC from clustering; show on map only.

In [ ]:
df_sdcc    = df[df["traffic_source"] == "SDCC_2024"].copy()
df_cluster = df[df["traffic_source"] != "SDCC_2024"].copy().reset_index(drop=True)

print(f"Clustering dataset : {len(df_cluster)} rows (DCC + DLR)")
print(f"SDCC display-only  : {len(df_sdcc)} rows")
print(f"\nDCC/DLR mean traffic : {df_cluster['traffic_volume'].mean():.1f}")
print(f"SDCC mean traffic    : {df_sdcc['traffic_volume'].mean():.1f}")

## 3. Feature Engineering — Gap Score

`gap_score = traffic_volume / (charger_count_nearby + 1)`

Higher score = high demand, low existing supply = priority candidate.

In [ ]:
df_cluster["gap_score"] = (
    df_cluster["traffic_volume"] / (df_cluster["charger_count_nearby"] + 1)
)
print("Gap Score statistics:")
print(df_cluster["gap_score"].describe())

## 4. Select Top 20% Candidates by Gap Score

In [ ]:
threshold = df_cluster["gap_score"].quantile(0.80)
df_priority = df_cluster[df_cluster["gap_score"] >= threshold].copy().reset_index(drop=True)

print(f"Gap score threshold (80th percentile): {threshold:.2f}")
print(f"Candidate sites selected: {len(df_priority)} / {len(df_cluster)}")

## 5. Elbow Method — Select Optimal K

In [ ]:
FEATURES = ["traffic_volume", "charger_count_nearby", "road_density", "lat", "lon"]
# ev_penetration_proxy excluded: constant 0.049, zero spatial variance

X = df_priority[FEATURES].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

inertias = []
sil_scores = []
K_range = range(2, 16)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(K_range, inertias, "bo-", linewidth=2, markersize=8)
ax1.set_xlabel("Number of Clusters (K)")
ax1.set_ylabel("Inertia")
ax1.set_title("Elbow Method")
ax1.grid(True, alpha=0.3)
ax2.plot(K_range, sil_scores, "rs-", linewidth=2, markersize=8)
ax2.set_xlabel("Number of Clusters (K)")
ax2.set_ylabel("Silhouette Score")
ax2.set_title("Silhouette Score by K")
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("elbow_silhouette_v2.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nSilhouette scores by K:")
for k, s in zip(K_range, sil_scores):
    print(f"  K={k:2d}: {s:.4f}")

## 6. Final K-Means (K=10)

In [ ]:
K = 10
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
df_priority["cluster"] = kmeans.fit_predict(X_scaled)

score = silhouette_score(X_scaled, df_priority["cluster"])
print(f"Silhouette Score (K={K}): {score:.4f}")
print(f"\nComparison:")
print(f"  Interim v1 (223 sites, DLR only) : 0.4235")
print(f"  v2         (974 sites, DCC+DLR)  : {score:.4f}")
print(f"\nNote: Score decrease expected — 4x more sites, larger geographic area.")
print(f"\nCluster sizes:")
print(df_priority["cluster"].value_counts().sort_index())

## 7. Select Top Recommendation per Cluster

In [ ]:
cluster_centers = (
    df_priority
    .sort_values("gap_score", ascending=False)
    .groupby("cluster")
    .first()
    .reset_index()
)
cluster_centers = cluster_centers.sort_values("gap_score", ascending=False).reset_index(drop=True)
cluster_centers["rank"] = cluster_centers.index + 1

print("Top 10 Recommendations:")
cols = ["rank","lat","lon","cluster","gap_score","traffic_volume","charger_count_nearby","road_density"]
print(cluster_centers[cols].to_string(index=False))

## 8. Export recommendations.csv

In [ ]:
out_cols = ["rank","lat","lon","cluster","gap_score","traffic_volume","charger_count_nearby","road_density","traffic_source"]
cluster_centers[out_cols].to_csv("recommendations.csv", index=False)
print("Saved: recommendations.csv")

## 9. Folium Map

In [ ]:
COLORS = [
    "red", "blue", "green", "purple", "orange",
    "darkred", "cadetblue", "darkgreen", "darkpurple", "darkblue"
]

m = folium.Map(location=[53.33, -6.25], zoom_start=11)

# All DCC+DLR sites (grey)
for _, row in df_cluster.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=3, color="lightgrey", fill=True, fill_opacity=0.3
    ).add_to(m)

# SDCC sites (blue outline, display only)
for _, row in df_sdcc.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=4, color="steelblue", fill=False, weight=1.5,
        popup=f"SDCC (display only)<br>Traffic: {row['traffic_volume']:.0f}"
    ).add_to(m)

# Priority candidates (coloured by cluster)
for _, row in df_priority.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=5, color=COLORS[int(row["cluster"])], fill=True, fill_opacity=0.5,
        popup=f"Traffic: {row['traffic_volume']:.0f}<br>Gap Score: {row['gap_score']:.1f}<br>Cluster: {int(row['cluster'])}"
    ).add_to(m)

# Recommendations (red bolt markers)
for _, row in cluster_centers.iterrows():
    folium.Marker(
        location=[row["lat"], row["lon"]],
        popup=f"Rank #{int(row['rank'])}<br>Traffic: {row['traffic_volume']:.0f}<br>Gap Score: {row['gap_score']:.1f}",
        icon=folium.Icon(color="red", icon="bolt", prefix="fa")
    ).add_to(m)

m.save("kmeans_recommendations_v2.html")
print("Map saved: kmeans_recommendations_v2.html")
m

## 10. Summary

In [ ]:
print("=== K-Means Clustering Summary (v2) ===")
print(f"Total SCATS sites loaded     : {len(df)}")
print(f"SDCC excluded (display only) : {len(df_sdcc)}")
print(f"Clustering dataset           : {len(df_cluster)} (DCC + DLR)")
print(f"Priority sites (top 20%)     : {len(df_priority)}")
print(f"Number of clusters (K)       : {K}")
print(f"Silhouette Score             : {score:.4f}")
print(f"Recommended locations        : {len(cluster_centers)}")
print()
print("Top 3 recommendations:")
for _, row in cluster_centers.head(3).iterrows():
    print(f"  Rank #{int(row['rank'])}: ({row['lat']:.4f}, {row['lon']:.4f}) — Gap Score: {row['gap_score']:.1f}")